# (1) Metadata Preparation

## The Speech Accent Archive

The Speech Accent Archive is a collection of spoken english hosted by George Mason University. Over 2000 speakers representing over 100 native languages read the following paragraph in English:

***Please call Stella. Ask her to bring these things with her from the store: Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a snack for her brother Bob. We also need a small plastic snake and a big toy frog for the kids. She can scoop these things into three red bags, and we will go meet her Wednesday at the train station.***

This makes the dataset very ideal for studying accents, and the recording quality for all samples is almost the same across all speakers. Because of this particular reason, we will instead use it to test whether our age detection models can accurately estimate a speaker's age. Using python code and dataframes from the [accent classifier repository](https://github.com/dwww2012/Accent-Classifier), I managed to download audio samples from the speech accent archive for the following native speakers:
 * English
 * Spanish
 * Arabic

From here, we will need compile all of the dataframes into one metadata file.

In [1]:
import os
import pandas as pd

filenames = os.listdir("./datasets/speech_accent_archive/metadata/")
filenames = [
    filename 
    for filename in filenames 
    if filename != '.ipynb_checkpoints'
]

metadata_path = "./datasets/speech_accent_archive/metadata/%s"
speaker_dataframes = []
for filename in filenames:
    speaker_dataframe = pd.read_csv(metadata_path % filename)
    
    #Remove unecessary columns
    speaker_dataframe.drop(
        columns=["Unnamed: 0", "Unnamed: 0.1", "speakerid"],
        inplace=True
    )
    
    #Rename columns
    speaker_dataframe.rename(
        columns={"sex": "gender", "age_onset": "age_english_onset", "native_language": "accent"},
        inplace=True
    )
    
    #Rearrange columns
    speaker_dataframe = speaker_dataframe[
        [
            "filename",
            "gender",
            "birthplace",
            "country",
            "age_english_onset",
            "accent",
            "age"
        ]
    ]
    
    speaker_dataframes.append(speaker_dataframe)
    
metadata_df = pd.concat(speaker_dataframes, ignore_index=True)
metadata_df.filename = metadata_df.filename.apply(lambda filename: filename + '.mp3')

metadata_df.to_csv("./datasets/speech_accent_archive/metadata.csv", index=False)
metadata_df

,filename,gender,birthplace,country,age_english_onset,accent,age
0,arabic10.mp3,male,"cairo, egypt",egypt,5.0,arabic,26.0
1,arabic12.mp3,male,"baghdad, iraq",iraq,11.0,arabic,32.0
2,arabic13.mp3,male,"zabbougha, lebanon",lebanon,15.0,arabic,25.0
3,arabic2.mp3,male,"damascus, syria",syria,2.5,arabic,18.0
4,arabic3.mp3,male,"doha, qatar",qatar,9.0,arabic,24.0
...,...,...,...,...,...,...,...
838,english572.mp3,male,"fort worth, texas, usa",usa,0.0,english,51.0
839,english573.mp3,male,"painesville, ohio, usa",usa,0.0,english,46.0
840,english575.mp3,male,"great falls, virginia, usa",usa,0.0,english,24.0
841,english578.mp3,male,"salford, lancashire, uk",uk,0.0,english,60.0


Let's check the number files in the clips directory, and the number of files in the dataframe, to see if they are the same. That way we can check if we have metadata for each file in our clips directory.

In [2]:
dataframe_list = list(metadata_df.filename)

directory_list = os.listdir("./datasets/speech_accent_archive/clips/")
print("Number of Rows in Dataframe: ", len(dataframe_list))
print("Number of Files in Directory: ", len(directory_list))

a = set(dataframe_list)
b = set(directory_list)

Number of Rows in Dataframe:  843
Number of Files in Directory:  1084


As you can see, the number of files in our clips directory is greater than the number of files listed in the dataframe. This means that there a 241 files in the directory that don't have any metadata. Naturally, will need to collect it for each 241 files.

In [3]:
import requests

from  bs4 import BeautifulSoup
from tqdm import tqdm

main_url = 'https://accent.gmu.edu/{}'
language_url = 'https://accent.gmu.edu/browse_language.php?function=find&language={}'

filenames = []
for filename in directory_list:
    if filename not in dataframe_list:
        filenames.append(filename[:-4])
        
ages = []
accents = []
genders = []
birthplaces = []
age_english_onsets = []

for filename in tqdm(filenames, desc='Progress'):
    language = ''.join(
        character 
        for character in filename
        if not character.isdigit()
    )
    #Extract the link to the speaker from the filename (name of file without extension)
    html = requests.get(language_url.format(language))
    soup = BeautifulSoup(html.content, 'html.parser')
    
    main_content = soup.find_all('div', attrs={'class': 'content'})[0]
    items = main_content.find_all('p')
    
    for item in items:
        link = item.find('a').get('href')
        linkname = item.text.split(", ")[0]
        
        if filename == linkname:
            speaker_url = main_url.format(link)
            break
    
    #From the link of the speaker, collect biographical data
    html = requests.get(speaker_url)
    soup = BeautifulSoup(html.content, 'html.parser')
        
    speaker_content = soup.find_all('div', attrs={'class': 'content'})[0]
    biography = soup.find_all('ul', attrs={'class': 'bio'})[0]
    biography_items = biography.find_all('li')
    
    age = float(biography_items[3].text.split()[2].strip(','))
    accent = str(biography_items[1].text.split()[2])
    gender = str(biography_items[3].text.split()[3].strip())
    birthplace = str(biography_items[0].text)[13:-6]
    age_english_onset = float(biography_items[4].text.split()[4].strip())
    
    #Collect metadata features and place them into their corresponding lists
    ages.append(age)
    accents.append(accent)
    genders.append(gender)
    birthplaces.append(birthplace)
    age_english_onsets.append(age_english_onset)
    
countries = [birthplace.split(", ")[-1] for birthplace in birthplaces]
filenames = [filename + '.mp3' for filename in filenames]
missing_metadata_df = pd.DataFrame(
    {
        'filename': filenames,
        'gender': genders,
        'birthplace': birthplaces,
        'country': countries,
        'age_english_onset': age_english_onsets,
        'accent': accents,
        'age': ages
    }
)
missing_metadata_df

Progress: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 241/241 [02:54<00:00,  1.38it/s]


,filename,gender,birthplace,country,age_english_onset,accent,age
0,arabic156.mp3,female,"cairo, egypt",egypt,6.0,arabic,18.0
1,spanish197.mp3,female,"madrid, spain",spain,3.0,spanish,20.0
2,spanish214.mp3,female,"neiva, colombia",colombia,26.0,spanish,45.0
3,spanish169.mp3,female,"ponce, puerto rico",puerto rico,10.0,spanish,36.0
4,arabic108.mp3,female,"riyadh, saudi arabia",saudi arabia,10.0,arabic,30.0
...,...,...,...,...,...,...,...
236,english601.mp3,male,"fairfax, virginia, usa",usa,0.0,english,49.0
237,english609.mp3,female,"narrandera, new south wales, australia",australia,0.0,english,20.0
238,spanish234.mp3,male,"maracaibo, venezuela",venezuela,13.0,spanish,51.0
239,arabic165.mp3,female,"beirut, lebanon",lebanon,12.0,arabic,62.0


Let's save the dataframe as a csv file, and check if we have metadata for each file.

In [4]:
metadata_df = pd.concat([metadata_df, missing_metadata_df])
metadata_df.to_csv("./datasets/speech_accent_archive/metadata.csv", index=False)
metadata_df

,filename,gender,birthplace,country,age_english_onset,accent,age
0,arabic10.mp3,male,"cairo, egypt",egypt,5.0,arabic,26.0
1,arabic12.mp3,male,"baghdad, iraq",iraq,11.0,arabic,32.0
2,arabic13.mp3,male,"zabbougha, lebanon",lebanon,15.0,arabic,25.0
3,arabic2.mp3,male,"damascus, syria",syria,2.5,arabic,18.0
4,arabic3.mp3,male,"doha, qatar",qatar,9.0,arabic,24.0
...,...,...,...,...,...,...,...
236,english601.mp3,male,"fairfax, virginia, usa",usa,0.0,english,49.0
237,english609.mp3,female,"narrandera, new south wales, australia",australia,0.0,english,20.0
238,spanish234.mp3,male,"maracaibo, venezuela",venezuela,13.0,spanish,51.0
239,arabic165.mp3,female,"beirut, lebanon",lebanon,12.0,arabic,62.0


In [5]:
print(sorted(directory_list) == sorted(list(metadata_df.filename)))

True


## Common Voice

The Common Voice dataset (Common Voice Corpus 11.0) is a very lage dataset that is designed for training speech enabled applications. Details can be found in this [link](https://commonvoice.mozilla.org/en/datasets). After downloading and uzipping the tar file, you will have metadata in the form of *tsv* files, and about **2.1 million** audio samples.

In [6]:
import os
import pandas as pd

filenames = os.listdir('./datasets/common_voice/metadata/')
filenames = [
    filename 
    for filename in filenames 
    if filename != '.ipynb_checkpoints' and filename != 'reported.tsv'
]
metadata_path = "./datasets/common_voice/metadata/%s"
speaker_dataframes = []
for filename in filenames:
    speaker_dataframe = pd.read_csv(metadata_path % filename, sep='\t')

    #Remove unncessary columns
    speaker_dataframe.drop(
        columns=["client_id", "sentence", "segment"],
        inplace=True
    )
    
    #Rename columns
    speaker_dataframe.rename(
        columns={"path": "filename", "accents": "accent"},
        inplace=True
    )
    
    #Rearrange columns
    speaker_dataframe = speaker_dataframe[
        [
            "filename",
            "up_votes",
            "down_votes",
            "locale",
            "gender",
            "accent",
            "age"
        ]
    ]
    
    speaker_dataframes.append(speaker_dataframe)
    
metadata_df = pd.concat(speaker_dataframes, ignore_index=True)
metadata_df.to_csv("./datasets/common_voice/metadata.csv", index=False)
metadata_df

/home/xnell90/anaconda3/envs/tf2-gpu/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (9) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


,filename,up_votes,down_votes,locale,gender,accent,age
0,common_voice_en_27710027.mp3,3,1,en,NaN,NaN,NaN
1,common_voice_en_699711.mp3,2,1,en,NaN,NaN,NaN
2,common_voice_en_21953345.mp3,3,2,en,NaN,NaN,NaN
3,common_voice_en_18132047.mp3,2,0,en,NaN,NaN,NaN
4,common_voice_en_27340672.mp3,2,0,en,NaN,NaN,NaN
...,...,...,...,...,...,...,...
3142384,common_voice_en_34925852.mp3,0,0,en,female,United States English,thirties
3142385,common_voice_en_34925854.mp3,0,0,en,female,United States English,thirties
3142386,common_voice_en_34925850.mp3,0,0,en,female,United States English,thirties
3142387,common_voice_en_34925856.mp3,0,0,en,female,United States English,thirties


Since we care mostly about predicting the speaker's age, let us find samples where we have labelled age.

In [7]:
metadata_df = metadata_df[metadata_df.age.notna()]
metadata_df

,filename,up_votes,down_votes,locale,gender,accent,age
14,common_voice_en_33532190.mp3,2,0,en,male,NaN,twenties
15,common_voice_en_18295850.mp3,2,0,en,male,NaN,twenties
18,common_voice_en_32371106.mp3,2,0,en,male,NaN,sixties
31,common_voice_en_32642818.mp3,2,0,en,female,NaN,seventies
64,common_voice_en_22338655.mp3,3,1,en,female,Hong Kong English,twenties
...,...,...,...,...,...,...,...
3142384,common_voice_en_34925852.mp3,0,0,en,female,United States English,thirties
3142385,common_voice_en_34925854.mp3,0,0,en,female,United States English,thirties
3142386,common_voice_en_34925850.mp3,0,0,en,female,United States English,thirties
3142387,common_voice_en_34925856.mp3,0,0,en,female,United States English,thirties


Remove duplicate filenames since there are duplicate filenames.

In [8]:
metadata_df.drop_duplicates(subset=['filename'], inplace=True)

/home/xnell90/anaconda3/envs/tf2-gpu/lib/python3.9/site-packages/pandas/util/_decorators.py:311: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return func(*args, **kwargs)


Now let's check if each file in our directory has metadata that can be obtained from the above dataframe.

In [9]:
dataframe_filenames = set(metadata_df.filename)
directory_filenames = set(os.listdir('./datasets/common_voice/clips/'))

common_filenames = dataframe_filenames.intersection(directory_filenames)

metadata_df = metadata_df[metadata_df.filename.isin(common_filenames)]
metadata_df.to_csv("./datasets/common_voice/metadata.csv", index=False)

metadata_df

,filename,up_votes,down_votes,locale,gender,accent,age
14,common_voice_en_33532190.mp3,2,0,en,male,NaN,twenties
15,common_voice_en_18295850.mp3,2,0,en,male,NaN,twenties
18,common_voice_en_32371106.mp3,2,0,en,male,NaN,sixties
31,common_voice_en_32642818.mp3,2,0,en,female,NaN,seventies
64,common_voice_en_22338655.mp3,3,1,en,female,Hong Kong English,twenties
...,...,...,...,...,...,...,...
3142384,common_voice_en_34925852.mp3,0,0,en,female,United States English,thirties
3142385,common_voice_en_34925854.mp3,0,0,en,female,United States English,thirties
3142386,common_voice_en_34925850.mp3,0,0,en,female,United States English,thirties
3142387,common_voice_en_34925856.mp3,0,0,en,female,United States English,thirties


In [10]:
len(directory_filenames)

2161670

As you can see, the number of files in our clips directory is greater than the number of files listed in the above dataframe. This means that we have files (~800,000) in our directory that don't have metadata. Because of the size of our dataset, let's ignore them for a moment. Instead, we will keep a list of those files for reference.

In [11]:
missing_filenames = list(directory_filenames - dataframe_filenames)
with open("./datasets/common_voice/missing_metadata.txt", "w") as missing_metadata:
    for missing_filename in missing_filenames:
        missing_metadata.write(str(missing_filename) + '\n')